# bias-correction-divide — faded example 3: Complete the first-moment correction inside an Adam step

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `bias-correction-divide`. Running the beacon reports progress on the `Optimizer: Adam bias-correction divide` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Adam bias-correction divide` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bias-correction-divide`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bias-correction-divide"
DD_SUBTOPIC = "Optimizer: Adam bias-correction divide"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A full Adam update corrects both moments with the SAME step `t` but DIFFERENT betas: `m_hat = m / (1 - beta1**t)` and `v_hat = v / (1 - beta2**t)`, then steps by `lr * m_hat / (sqrt(v_hat) + eps)`. The first-moment correction must use `beta1`, not `beta2`.

## Faded exercise 3

### Faded — first-moment correction in the Adam update

Implement `adam_update(m, v, beta1, beta2, t_step, lr, eps)`. The second-moment correction and the final update assembly are provided. Fill in the first-moment bias correction `m_hat`.

**Fill in:** the bias-corrected first moment `m / (1 - beta1 ** t_step)`

In [ ]:
def adam_update(m, v, beta1, beta2, t_step, lr=1e-3, eps=1e-8):
    m_hat = m / (1 - beta1 ** t_step)
    v_hat = v / (1 - beta2 ** t_step)
    return lr * m_hat / (v_hat.sqrt() + eps)


def _test():
    m = t.tensor([0.01, -0.02, 0.03])
    v = t.tensor([0.0001, 0.0004, 0.0009])
    beta1, beta2, t_step = 0.9, 0.999, 1
    lr, eps = 1e-3, 1e-8
    out = adam_update(m, v, beta1, beta2, t_step, lr, eps)
    m_hat = m / (1 - beta1 ** t_step)
    v_hat = v / (1 - beta2 ** t_step)
    expected = lr * m_hat / (v_hat.sqrt() + eps)
    assert out.shape == m.shape
    assert t.allclose(out, expected, atol=1e-10), (out, expected)
    # sign of update follows sign of m
    assert bool(t.all(t.sign(out) == t.sign(m)))


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def adam_update(m, v, beta1, beta2, t_step, lr=1e-3, eps=1e-8):
    m_hat = m / (1 - beta1 ** t_step)
    v_hat = v / (1 - beta2 ** t_step)
    return lr * m_hat / (v_hat.sqrt() + eps)
```
</details>